# Lab: Bring your own custom container with Amazon SageMaker

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This Immersion Day has been tested using the following SageMaker Distribution images:

<ul>
<li><strong>SageMaker Distribution Image 3.7.0</strong></li>
    <li><strong>SageMaker Distribution Image 3.8.0</strong></ul>
</ul>  
and the following SageMaker Python SDK version
<ul>
    <li><strong>SageMaker Python SDK version 3.4.0</strong></li>
</ul>
</div>


## Overview

### Background
Here, we'll show how to bring your docker cotainer that packages your environment and code. We showcase the [decision tree](http://scikit-learn.org/stable/modules/tree.html) algorithm from the widely used [scikit-learn](http://scikit-learn.org/stable/) machine learning package. The example is purposefully fairly trivial since the point is to show the surrounding structure that you'll want to add to your own container so you can bring it to Amazon SageMaker for training and hosting.


### High-level overview

The following diagram shows how you typically train and deploy a model with Amazon SageMaker:

<div>
<img src="https://docs.aws.amazon.com/sagemaker/latest/dg/images/sagemaker-architecture.png" width="900"/>
</div>

The area labeled SageMaker highlights the two components of SageMaker: model training and model deployment. The area labeled [EC2 container registry](https://aws.amazon.com/ecr/) is where we store, manage, and deploy our Docker container images. The training data and model artifacts are stored in S3 bucket. 

In this lab, we use a single image to support both model training and hosting for simplicity. Sometimes you’ll want separate images for training and hosting because they have different requirements. 

The high-level steps include:
1. **Building the container** - We walk through the different components of the containers and inspect the docker file. Then we build and push the container to ECR. 
2. **Setup & Upload Data** - Once our container is built and registered. We ready sagemaker and upload the data to S3. 
3. **Model Training** - Create a training job using SageMaker Python SDK. It will pull data from S3 and use the container we built.  
4. **Model Deployment** - Once training is complete, deploy our model to a HTTP endpoint using SageMaker Python SDK. 
5. **Run Inferences** - Run predictions to test our model.
6. **Cleanup**



## Building the container
[Docker](https://aws.amazon.com/docker/#:~:text=Docker%20is%20a%20software%20platform,test%2C%20and%20deploy%20applications%20quickly.&text=Running%20Docker%20on%20AWS%20provides,distributed%20applications%20at%20any%20scale.) packages software into standardized units called [containers](https://aws.amazon.com/containers/) that have everything the software needs to run including libraries, system tools, code, and runtime. Using Docker, you can quickly deploy and scale applications into any environment and know your code will run.


Amazon SageMaker uses Docker to allow users to train and deploy arbitrary algorithms. More details on [how to use docker containers with sagemaker](https://docs.aws.amazon.com/sagemaker/latest/dg/docker-containers.html).

### Walkthrough of the container directory
You can find the source code of the sample container we are using in [this GitHub repository](https://github.com/aws/amazon-sagemaker-examples/tree/main/advanced_functionality/scikit_bring_your_own). 

The container directory contains all the components you need to package for SageMaker:

```
.
|-- Dockerfile
`-- decision_trees
    |-- nginx.conf
    |-- predictor.py
    |-- serve
    |-- train
    `-- wsgi.py
```

Let’s discuss each of these in turn:

- `Dockerfile` describes how to build your Docker container image. More details below. 
- `decision_trees` is the directory which contains the files that will be installed in the container.

In this simple application, we only install five files in the container. These five show the standard structure of our Python containers, although you are free to choose a different toolset or programming language and therefore could have a different layout.

The files that we’ll put in the container are:

- `nginx.conf` is the configuration file for the nginx front-end. Generally, you should be able to take this file as-is.
- `predictor.py` is the program that actually implements the Flask web server and the decision tree predictions for this app. You’ll want to customize the actual prediction parts to your application. Since this algorithm is simple, we do all the processing here in this file, but you may choose to have separate files for implementing your custom logic.
- `serve` is the program started when the container is started for hosting. It simply launches the gunicorn server which runs multiple instances of the Flask app defined in predictor.py. You should be able to take this file as-is.
- `train` is the program that is invoked when the container is run for training. You will modify this program to implement your training algorithm.
- `wsgi.py` is a small wrapper used to invoke the Flask app. You should be able to take this file as-is.

In summary, the two files you will probably want to change for your application are `train` and `predictor.py`

### Install packages
Please choose `Python 3 (ipykernel)` kernel to proceed.

We will first install the prerequisite packages. They should be already installed if you run the [Setup.ipynb](../Setup.ipynb) notebook. If you experience problems, uncomment the following three cells to re-install the right libraires and the restart the kernel and then continue

In [ ]:
# !pip install --root-user-action=ignore --upgrade pip
# !pip install --root-user-action=ignore -q pandas==2.1.4
# !pip install --root-user-action=ignore -q awswrangler==3.5.1 --no-cache

In [ ]:
## Uncomment and execute if having errors import any of the sagemaker libraries
#!pip install -U "sagemaker==3.4.0" --force-reinstall

In [ ]:
# # Restart kernel to get the packages
# import IPython
# IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import sagemaker
import sagemaker.core
import sagemaker.train
import sagemaker.serve
import sagemaker.mlops
from importlib.metadata import version

print(f"sagemaker: {version('sagemaker')}")
print(f"sagemaker.core: {version('sagemaker.core')}")
print(f"sagemaker.train: {version('sagemaker.train')}")
print(f"sagemaker.server: {version('sagemaker.serve')}")
print(f"sagemaker.mlops: {version('sagemaker.mlops')}")

### Install Docker

To use Docker, you must manually install it from the terminal of your JupyterLab application. Please get familiar with the docker operations that are currently supported in Studio [see here](https://docs.aws.amazon.com/sagemaker/latest/dg/studio-updated-local.html).


In [ ]:
%%bash

# see https://docs.docker.com/engine/install/ubuntu/#install-using-the-repository
sudo apt-get update
sudo apt-get install -y ca-certificates curl
sudo install -m 0755 -d /etc/apt/keyrings
sudo curl -fsSL https://download.docker.com/linux/ubuntu/gpg -o /etc/apt/keyrings/docker.asc
sudo chmod a+r /etc/apt/keyrings/docker.asc

# Add the repository to Apt sources:
echo \
  "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/docker.asc] https://download.docker.com/linux/ubuntu \
  $(. /etc/os-release && echo "$VERSION_CODENAME") stable" | \
  sudo tee /etc/apt/sources.list.d/docker.list > /dev/null
sudo apt-get update

## Currently only Docker version 20.10.X is supported in Studio: see https://docs.aws.amazon.com/sagemaker/latest/dg/studio-updated-local.html
# pick the latest patch from:
# apt-cache madison docker-ce | awk '{ print $3 }' | grep -i 20.10
VERSION_STRING=5:20.10.24~3-0~ubuntu-jammy
sudo apt-get install docker-ce-cli=$VERSION_STRING docker-compose-plugin -y

# validate the Docker Client is able to access Docker Server at [unix:///docker/proxy.sock]
docker version
# validate the Docker Compose plugin installed
docker compose version

# Install Compose Switch. this is a replacement to the Compose V1 docker-compose (python) executable.
# It translates the command line into Compose V2 docker compose then run the latter.
# this is needed by the sagemaker python sdk to run in local mode
# see https://github.com/docker/compose-switch
curl -fL https://raw.githubusercontent.com/docker/compose-switch/master/install_on_linux.sh | sudo sh
docker-compose version

### The Dockerfile
The `Dockerfile` describes the image that we want to build. You can think of it as describing the complete operating system installation of the system that you want to run. A Docker container running is quite a bit lighter than a full operating system, however, because it takes advantage of Linux on the host machine for the basic operations.

For the Python science stack, we will start from a standard Ubuntu installation and run the normal tools to install the things needed by `scikit-learn`. Finally, we add the code that implements our specific algorithm to the container and set up the right environment to run under.

Let's take a look of what's inside our `Dockerfile`:

In [ ]:
!pygmentize lab03_container/Dockerfile

### Building and registering the container

In [ ]:
%%sh
# Login to ECR
aws --region ${AWS_DEFAULT_REGION} ecr get-login-password | docker login --username AWS --password-stdin ${AWS_ACCOUNT_ID}.dkr.ecr.${AWS_DEFAULT_REGION}.amazonaws.com/sagemaker-decision-trees

# If the repository doesn't exist in ECR, create it.
aws ecr describe-repositories --repository-names "sagemaker-decision-trees" > /dev/null 2>&1

if [ $? -ne 0 ]
then
    aws ecr create-repository --repository-name "sagemaker-decision-trees" > /dev/null
fi

cd lab03_container

chmod +x decision_trees/train
chmod +x decision_trees/serve

# Build the image - it might take a few minutes to complete this step
docker build --network sagemaker . -t ${AWS_ACCOUNT_ID}.dkr.ecr.${AWS_DEFAULT_REGION}.amazonaws.com/sagemaker-decision-trees:latest
# Push the image to ECR
docker push ${AWS_ACCOUNT_ID}.dkr.ecr.${AWS_DEFAULT_REGION}.amazonaws.com/sagemaker-decision-trees:latest

## Setup & Upload Data

### Setup the Environment 
Here we specify a bucket to use and the role that will be used for working with SageMaker.



In [ ]:
S3_prefix = "DEMO-scikit-byo-iris"

# Define IAM role
import boto3
import re

import os
import numpy as np
import pandas as pd
from sagemaker.core.helper.session_helper import get_execution_role

role = get_execution_role()

The session remembers our connection parameters to SageMaker. We’ll use it to perform all of our SageMaker operations.

In [ ]:
from sagemaker.core.helper.session_helper import Session
from time import gmtime, strftime

sess = Session()

### Upload data to S3 Bucket

When training large models with huge amounts of data, you’ll typically use big data tools, like Amazon Athena, AWS Glue, or Amazon EMR, to create your data in S3. For the purposes of this example, we’re using some the [classic Iris dataset](https://en.wikipedia.org/wiki/Iris_flower_data_set) in the `lab03_data` directory. 

We can use use the tools provided by the [SageMaker Python SDK](https://sagemaker.readthedocs.io/en/stable/) to upload the data to a default bucket.

In [ ]:
# cell 05

WORK_DIRECTORY = "lab03_data"

data_location = sess.upload_data(WORK_DIRECTORY, key_prefix=S3_prefix)

## Model Training

In SageMaker Python SDK V3, we use `ModelTrainer` to define how to use the container to train. This includes the configuration we need to invoke SageMaker training:

- `training_image (str)` - The [Amazon Elastic Container Registry](https://aws.amazon.com/ecr/) path where the docker image is registered.
- `role (str)` - SageMaker IAM role as obtained above.
- `compute (Compute)` - Compute configuration with `instance_type` and `instance_count`.
- `output_data_config (dict)` - Configuration with `s3_output_path` where the model artifact will be written.
- `sagemaker_session (Session)` - the SageMaker session object.

Then we use `model_trainer.train()` method to train against the data that we uploaded.
The API calls the Amazon SageMaker `CreateTrainingJob` API to start model training. Input data is specified using `InputData` objects with a `channel_name` and `data_source`.

In [ ]:
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.train.configs import Compute, InputData

account = sess.boto_session.client("sts").get_caller_identity()["Account"]
region = sess.boto_session.region_name
image_uri = "{}.dkr.ecr.{}.amazonaws.com/sagemaker-decision-trees:latest".format(account, region)

tree = ModelTrainer(
    training_image=image_uri,
    role=role,
    compute=Compute(instance_type="ml.c5.2xlarge", instance_count=1),
    output_data_config={"s3_output_path": "s3://{}/output".format(sess.default_bucket())},
    sagemaker_session=sess,
    hyperparameters={
        "max_leaf_nodes": "2"
    },
)

file_location = data_location + "/iris.csv"
tree.train(input_data_config=[InputData(channel_name="training", data_source=file_location)])

## Model Deployment
You can use a trained model to get real time predictions using HTTP endpoint. Follow these steps to walk you through the process.

In SageMaker Python SDK V3, we use `ModelBuilder` to create a deployable model. After training completes, we:
1. Get the model artifacts location from `tree._latest_training_job.model_artifacts.s3_model_artifacts`
2. Create a `SchemaBuilder` with sample input/output for serialization
3. Create a `ModelBuilder` with the image URI, model data URL, and schema
4. Call `build()` to prepare the model, then `deploy()` to create the endpoint

The `deploy()` method returns an `Endpoint` object (not a Predictor like in V2). Use `endpoint.invoke()` to make predictions.

Key parameters:
- `image_uri (str)` – The container image for inference.
- `s3_model_data_url (str)` – S3 path to the model artifacts from training.
- `schema_builder (SchemaBuilder)` – Defines input/output serialization.
- `role_arn (str)` – IAM role for the endpoint.
- `mode (Mode)` – Optional. Use `Mode.LOCAL_CONTAINER` for local testing.

In [ ]:
from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.serve.builder.schema_builder import SchemaBuilder

# Get model data from training job
model_data_url = tree._latest_training_job.model_artifacts.s3_model_artifacts

schema_builder = SchemaBuilder(
    sample_input="5.1,3.5,1.4,0.2",
    sample_output="setosa",
)

model_builder = ModelBuilder(
    image_uri=image_uri,
    s3_model_data_url=model_data_url,
    schema_builder=schema_builder,
    role_arn=role,
)

model = model_builder.build()

endpoint = model_builder.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
)


## Run Inferences


### Preparing test data
In order to do some predictions, we’ll extract some of the data we used for training and do predictions against it. This is, of course, bad statistical practice, but an easy way to see how the mechanism works.

In [ ]:
print(file_location)

In [ ]:
import awswrangler as wr

shape = wr.s3.read_csv(file_location, header=None)

# shape=pd.read_csv(file_location, header=None)
shape.sample(3)

In [ ]:
# drop the label column in the training set
shape.drop(shape.columns[[0]], axis=1, inplace=True)
shape.sample(3)

In [ ]:
import itertools

a = [50 * i for i in range(3)]
b = [40 + i for i in range(10)]
indices = [i + j for i, j in itertools.product(a, b)]

test_data = shape.iloc[indices[:-1]]

### Predictions

In V3, the `deploy()` method returns an `Endpoint` object. Use `endpoint.invoke()` with `body` (the data) and `content_type` to make predictions. Alternatively, you can use the boto3 `sagemaker-runtime` client directly with `invoke_endpoint()`.

In [ ]:
import io
import pandas as pd

# If test_data is a DataFrame
csv_data = test_data.to_csv(header=False, index=False)
result = endpoint.invoke(body=csv_data, content_type="text/csv")
print(result.body.read().decode("utf-8"))


In [ ]:
import boto3

runtime = boto3.client("sagemaker-runtime")
response = runtime.invoke_endpoint(
    EndpointName=endpoint.endpoint_name,
    ContentType="text/csv",
    Body=test_data.to_csv(header=False, index=False)
)
print(response["Body"].read().decode("utf-8"))

## Cleanup
After completing the lab, use these steps to [delete the endpoint through AWS Console](https://docs.aws.amazon.com/sagemaker/latest/dg/ex1-cleanup.html) or simply run the following code


In [ ]:
sess.delete_endpoint(endpoint.endpoint_name)

Remove the container artifacts and data we downloaded.